In [ ]:
import torch
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from google.colab import files
import cv2
from google.colab.patches import cv2_imshow

In [ ]:
class ConvBlock(torch.nn.Module):
    def __init__(self, in_c, out_c, kernel_size=3, stride=1):
        super().__init__()
        padding = kernel_size // 2
        self.conv = torch.nn.Conv2d(in_c, out_c, kernel_size, stride, padding, bias=False)
        self.bn = torch.nn.BatchNorm2d(out_c)
        self.act = torch.nn.LeakyReLU(0.1)
    def forward(self, x): return self.act(self.bn(self.conv(x)))

class ResidualBlock(torch.nn.Module):
    def __init__(self, in_c):
        super().__init__()
        self.layer = torch.nn.Sequential(
            ConvBlock(in_c, in_c // 2, 1),
            ConvBlock(in_c // 2, in_c, 3)
        )
    def forward(self, x): return x + self.layer(x)

class Darknet53(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = torch.nn.Sequential(
            ConvBlock(3, 32),
            ConvBlock(32, 64, 3, 2),
            ResidualBlock(64)
        )
        self.layer2 = torch.nn.Sequential(
            ConvBlock(64, 128, 3, 2),
            *[ResidualBlock(128) for _ in range(2)]
        )
        self.layer3 = torch.nn.Sequential(
            ConvBlock(128, 256, 3, 2),
            *[ResidualBlock(256) for _ in range(8)]
        )
        self.layer4 = torch.nn.Sequential(
            ConvBlock(256, 512, 3, 2),
            *[ResidualBlock(512) for _ in range(8)]
        )
        self.layer5 = torch.nn.Sequential(
            ConvBlock(512, 1024, 3, 2),
            *[ResidualBlock(1024) for _ in range(4)]
        )
    def forward(self, x):
        for l in [self.layer1, self.layer2, self.layer3, self.layer4, self.layer5]:
            x = l(x)
        return x

class AttentionModule(torch.nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.gamma = torch.nn.Parameter(torch.zeros(1))
    def forward(self, x):
        attn = torch.softmax(x.view(x.size(0), -1), dim=1).view_as(x)
        return x + self.gamma * attn * x

class AttentionYOLOv3(torch.nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.backbone = Darknet53()
        self.attn = AttentionModule(1024)
        self.gap = torch.nn.AdaptiveAvgPool2d((1, 1))
        self.fc = torch.nn.Linear(1024, num_classes)
    def forward(self, x):
        feat = self.backbone(x)
        feat = self.attn(feat)
        out = self.gap(feat).flatten(1)
        logits = self.fc(out)
        return logits

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AttentionYOLOv3(num_classes=2).to(device)

In [ ]:
_uploaded_files = files.upload()

In [ ]:
model.load_state_dict(torch.load("/content/attention_yolov3_drowsy_10epochs.pth", map_location=device))
model.eval()

In [ ]:
transform = transforms.Compose([
    transforms.Resize((416, 416)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
_uploaded_files = files.upload()


In [ ]:
video_path = "/content/d_1.mp4"
cap = cv2.VideoCapture(video_path)

In [ ]:
fourcc = cv2.VideoWriter_fourcc(*'XVID')
fps = cap.get(cv2.CAP_PROP_FPS)  # Get FPS from video
width = int(cap.get(3))
height = int(cap.get(4))
out = cv2.VideoWriter('output.avi', fourcc, 1, (width, height))  # 1 frame per second in output

classes = ["drowsy", "notdrowsy"]

frame_buffer = []
probs_buffer = []

frame_count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Convert frame for model
    img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    input_tensor = transform(img).unsqueeze(0).to(device)

    # Predict
    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)[0].cpu().numpy()  # store probabilities

    frame_buffer.append(frame)
    probs_buffer.append(probs)
    frame_count += 1

    # Process once per second
    if frame_count >= fps:
        avg_probs = np.mean(probs_buffer, axis=0)
        major_idx = np.argmax(avg_probs)
        label = f"{classes[major_idx]} ({avg_probs[major_idx]*100:.1f}%)"
        color = (0, 0, 255) if major_idx == 0 else (0, 255, 0)

        # Pick the frame that is most confident for the major class
        best_frame_idx = np.argmax([p[major_idx] for p in probs_buffer])
        rep_frame = frame_buffer[best_frame_idx].copy()

        # Draw label on representative frame
        cv2.putText(rep_frame, label, (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.1, color, 2)
        cv2_imshow(rep_frame)
        out.write(rep_frame)

        # Reset buffers for next second
        frame_buffer = []
        probs_buffer = []
        frame_count = 0

cap.release()
out.release()
cv2.destroyAllWindows()